# 06 — Concept Drift and Automated Retraining

**The failure this prevents.** Google Flu Trends was fitted on 2003–2008 data and
never refitted. Google's own search ranking changed 86 times in two months in
2012, silently altering the model's inputs. By 2013, media-driven panic searching
had GFT overestimating influenza by **140%**. Nobody noticed automatically,
because nothing was watching the residuals.

Something watches here (shortcoming #3, critical rule #4). This notebook shows:

1. the two detectors and what each is good at;
2. that they do **not** fire on a stable stream — a detector that cries wolf gets
   switched off, which is worse than having none;
3. the retraining gate: a refit is promoted only if it beats the incumbent on
   held-out data, so an automated loop cannot quietly degrade itself.

In [ ]:
# Make the repo importable regardless of where Jupyter was launched from.
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src").is_dir() and (p / "config").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import logging
logging.getLogger("afya").setLevel(logging.WARNING)   # keep notebook output readable

import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
print(f"repo root: {ROOT}")

In [ ]:
# Plotting is optional throughout these notebooks: matplotlib is not a hard
# dependency of AFYA-PREDICT, because the platform must install on low-spec
# district hardware. Every notebook falls back to printed tables without it.
#
# Backend selection matters more than it looks. Inside a Jupyter kernel,
# matplotlib configures its own inline backend and we leave it alone. Anywhere
# else - `nbconvert --execute`, CI, a headless server - a GUI backend will block
# forever on a window that never opens (a set-but-unreachable $DISPLAY is enough
# to trigger it), so we force the non-interactive Agg backend.
import os
import sys

try:
    import matplotlib
    _in_kernel = "ipykernel" in sys.modules
    if not os.environ.get("MPLBACKEND") and not _in_kernel:
        matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    plt.rcParams["figure.figsize"] = (11, 4)
    plt.rcParams["axes.grid"] = True
    plt.rcParams["grid.alpha"] = 0.3
    HAS_PLT = True
    print(f"matplotlib {matplotlib.__version__} on the "
          f"{matplotlib.get_backend()} backend")
except ImportError:
    HAS_PLT = False
    print("matplotlib not installed - tables will be printed instead of plotted")

## 1. The two detectors

**Page-Hinkley** tracks the cumulative deviation of each error from its running
mean, less a tolerance. When the gap between that cumulative sum and its own
minimum exceeds a threshold, a *sustained shift* has occurred — a model that has
started running consistently high.

**ADWIN** keeps a window of recent errors and tests every split point for a
significant difference in sub-window means, using a Hoeffding-style bound. It
needs no prior on the size of the change and adapts its own window.

They fail differently, which is why both run.

In [ ]:
from src.models.drift_detector import ADWIN, DriftDetector, PageHinkley, detect_drift

rng = np.random.default_rng(42)
N = 200
CHANGE_POINT = 120

stable = rng.normal(0.0, 1.0, N)

drifting = stable.copy()
drifting[CHANGE_POINT:] += 4.0            # a step change in the error mean

variance_shift = stable.copy()
variance_shift[CHANGE_POINT:] *= 4.0      # same mean, wider spread

gradual = stable + np.concatenate([np.zeros(CHANGE_POINT),
                                   np.linspace(0, 5, N - CHANGE_POINT)])

scenarios = {"stable": stable, "step change": drifting,
             "variance shift": variance_shift, "gradual drift": gradual}

rows = []
for name, residuals in scenarios.items():
    report = detect_drift(np.zeros(N), -residuals, min_observations=20)
    detectors = sorted({e["detector"] for e in report["events"]})
    first = min((e["index"] for e in report["events"]), default=None)
    rows.append({
        "scenario": name,
        "drift_detected": report["drift_detected"],
        "detectors": ", ".join(detectors) or "-",
        "first_flag_at": first,
        "delay_after_change": None if first is None else first - CHANGE_POINT,
        "events": len(report["events"]),
    })
display(pd.DataFrame(rows))
print(f"\nTrue change point: observation {CHANGE_POINT}")

The stable stream must show `drift_detected = False`. A detector that fires on
noise trains operators to ignore it, which reproduces the GFT outcome by a
different route.

In [ ]:
if HAS_PLT:
    fig, axes = plt.subplots(len(scenarios), 1, figsize=(12, 2.1 * len(scenarios)), sharex=True)
    for ax, (name, residuals) in zip(axes, scenarios.items()):
        ax.plot(residuals, lw=0.8, color="tab:blue")
        ax.axvline(CHANGE_POINT, color="k", ls="--", lw=1)
        report = detect_drift(np.zeros(N), -residuals, min_observations=20)
        for event in report["events"]:
            ax.axvline(event["index"], color="tab:red", ls=":", lw=1.4)
        ax.set_ylabel(name, fontsize=8)
    axes[-1].set_xlabel("observation")
    fig.suptitle("black dashed = true change point | red dotted = detector flag", y=0.995)
    plt.tight_layout(); plt.show()

### Detection delay versus false-alarm rate

In [ ]:
delays = []
for seed in range(25):
    local_rng = np.random.default_rng(seed)
    residuals = local_rng.normal(0, 1, N)
    residuals[CHANGE_POINT:] += 3.0
    report = detect_drift(np.zeros(N), -residuals, min_observations=20)
    if report["events"]:
        delays.append(min(e["index"] for e in report["events"]) - CHANGE_POINT)

false_alarms = 0
for seed in range(25):
    local_rng = np.random.default_rng(1000 + seed)
    report = detect_drift(np.zeros(N), -local_rng.normal(0, 1, N), min_observations=20)
    false_alarms += bool(report["drift_detected"])

print(f"Across 25 runs with a 3-sigma step at observation {CHANGE_POINT}:")
print(f"  detected in {len(delays)}/25 runs")
if delays:
    print(f"  median detection delay : {int(np.median(delays))} observations")
    print(f"  range                  : {min(delays)} - {max(delays)}")
print(f"\nFalse alarms on 25 stable streams: {false_alarms}/25")

## 2. Drift on a real model

Now the realistic case: a model fitted on one period, then applied to data whose
relationships have shifted. This simulates what actually happens in the field —
a change in reporting practice, a new case definition, a vector-control campaign
that alters the climate-to-case relationship.

In [ ]:
from src.core.config_loader import load_region_config, load_disease_config
from src.core.geo import subset_region

FULL_REGION = load_region_config("tanzania")
print(f"{len(FULL_REGION.districts)} councils across "
      f"{len({d.region for d in FULL_REGION.districts})} regions")

# A small, ecologically diverse subset keeps these notebooks fast to run.
# Swap in FULL_REGION for a national analysis (much slower).
STUDY_DISTRICTS = [
    "Kinondoni",     # dense coastal city
    "Ilala",         # dense coastal city, adjacent to Kinondoni
    "Mwanza City",   # lakeside city
    "Sengerema",     # rural lakeside, low WASH coverage
    "Dodoma City",   # semi-arid central
    "Songea MC",     # southern highlands
]
REGION = subset_region(FULL_REGION, STUDY_DISTRICTS)
pd.DataFrame([d.model_dump() for d in REGION.districts]).set_index("name")

In [ ]:
from src.data_ingestion.normalizer import ingest
from src.models.auto_retrain import _slice_weeks
from src.models.registry import build_module

module = build_module("malaria", region=REGION)
SOURCES = sorted(set(module.config.required_sources) | {"dhis2"})
panel = ingest(SOURCES, "2019-W01", "2024-W52", region=REGION)
matrix = module.build_feature_matrix(panel)

weeks = matrix.weeks
train_weeks = set(weeks[:int(len(weeks) * 0.6)])
monitor_weeks = [w for w in weeks if w not in train_weeks]

module.train(_slice_weeks(matrix, train_weeks))
print(f"trained on {len(train_weeks)} weeks, monitoring the next {len(monitor_weeks)}")

In [ ]:
from src.models.drift_detector import residual_series

monitor = _slice_weeks(matrix, set(monitor_weeks)).dropna_rows()
district = monitor.districts[0]
local = monitor.for_district(district)
model = module.model_for(district)

predicted = model.predict(local.X)
actual = local.y.to_numpy(dtype=float)
residuals = residual_series(actual, predicted)

print(f"district: {district}")
print(f"{len(residuals)} monitored residuals, "
      f"mean {residuals.mean():+.2f}, std {residuals.std():.2f}\n")

natural = detect_drift(actual, predicted, min_observations=20)
print(f"Drift on the genuine residual stream: {natural['drift_detected']}")
for event in natural["events"][:3]:
    print(f"  [{event['detector']}] at observation {event['index']}: {event['message']}")
if not natural["events"]:
    print("  (the data-generating process is stationary here, as expected)")

### Inject a reporting-practice change

A step change in reported counts — the kind a new case definition or a facility
reporting-policy change produces — is exactly what silently broke GFT.

In [ ]:
SHIFT_AT = len(actual) // 2
contaminated = actual.copy().astype(float)
contaminated[SHIFT_AT:] *= 1.8          # counts jump 80%: definition change

report = detect_drift(contaminated, predicted, min_observations=20)
print(f"Injected an 80% level shift at observation {SHIFT_AT} of {len(actual)}\n")
print(f"drift detected: {report['drift_detected']}")
for event in report["events"][:4]:
    print(f"  [{event['detector']:14}] observation {event['index']:>3} "
          f"(statistic {event['statistic']:.2f})")

if report["events"]:
    first = min(e["index"] for e in report["events"])
    print(f"\nCaught {first - SHIFT_AT} observations after the change.")
    print("GFT ran mis-specified for roughly three years.")

In [ ]:
if HAS_PLT:
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(actual, label="actual (original)", lw=1, color="k", alpha=0.5)
    ax.plot(contaminated, label="actual (after definition change)", lw=1.2, color="tab:red")
    ax.plot(predicted, label="model prediction", lw=1.2, color="tab:blue")
    ax.axvline(SHIFT_AT, ls="--", c="k", lw=1)
    for event in report["events"]:
        ax.axvline(event["index"], ls=":", c="tab:green", lw=1.4)
    ax.set_xlabel("monitored observation"); ax.set_ylabel("cases")
    ax.set_title("black dashed = injected change | green dotted = drift flag")
    ax.legend(fontsize=8)
    plt.show()

## 3. The retraining decision

`AutoRetrainer` combines two triggers:

* **cadence** — the disease config's `retrain_frequency` (monthly by default), so
  the model refits even when nothing looks wrong;
* **drift** — the residual monitor firing, which refits *immediately* rather than
  waiting for the next scheduled slot.

In [ ]:
from src.models.auto_retrain import RETRAIN_INTERVAL_DAYS, AutoRetrainer

retrainer = AutoRetrainer(module)
print(f"disease           : {module.config.name}")
print(f"retrain frequency : {module.config.model.retrain_frequency} "
      f"({RETRAIN_INTERVAL_DAYS[module.config.model.retrain_frequency]} days)")
print(f"rolling window    : {retrainer.rolling_window_weeks} weeks "
      f"({module.config.model.min_training_months} months minimum)\n")

n_recorded = retrainer.record_residuals(contaminated, predicted)
print(f"recorded {n_recorded} residuals to the monitored stream")

decision = retrainer.should_retrain(residuals=retrainer.monitored_residuals())
print(f"\nshould_retrain : {decision.should_retrain}")
for reason in decision.reasons:
    print(f"  - {reason}")
print(f"drift events   : {len(decision.drift_events)}")

## 4. The promotion gate

This is what stops an automated loop from degrading itself. The candidate model
is fitted on the latest rolling window and scored against the incumbent on a
holdout; **it is only promoted if it wins**. Otherwise the previous model keeps
serving and the attempt is recorded.

In [ ]:
import copy

before = {scope: m.version for scope, m in module.models.items()}
outcome = retrainer.retrain(panel, decision=decision, end_week=weeks[-1], holdout_weeks=12)

print(f"performed : {outcome.performed}")
print(f"promoted  : {outcome.promoted}")
print(f"incumbent holdout MAE : {outcome.incumbent_mae}")
print(f"candidate holdout MAE : {outcome.candidate_mae}")
print()
for reason in outcome.reasons:
    print(f"  - {reason}")

after = {scope: m.version for scope, m in module.models.items()}
changed = [s for s in after if before.get(s) != after[s]]
print(f"\nmodel versions changed for {len(changed)} scope(s)")
print("If not promoted, the served model is byte-identical to what it was before.")

### A candidate that is genuinely worse must be rejected

The gate is only meaningful if it actually blocks something. Here a deliberately
bad candidate — trained on a tiny, unrepresentative slice — is offered up.

In [ ]:
from src.models.base_model import TrainedModel

good_module = build_module("malaria", region=REGION)
good_module.train(_slice_weeks(matrix, set(weeks[:200])))
gate = AutoRetrainer(good_module)

holdout = _slice_weeks(matrix, set(weeks[-30:]))
incumbent_mae = gate._evaluate(good_module, holdout)

sabotaged = build_module("malaria", region=REGION)
sabotaged.train(_slice_weeks(matrix, set(weeks[:40])))     # far too little history
candidate_mae = gate._evaluate(sabotaged, holdout)

print(f"incumbent (200 weeks of training) holdout MAE : {incumbent_mae:.2f}")
print(f"candidate ( 40 weeks of training) holdout MAE : {candidate_mae:.2f}")
would_promote = candidate_mae is None or incumbent_mae is None or candidate_mae <= incumbent_mae
print(f"\nwould the gate promote the weaker candidate? {would_promote}")
print("The same comparison runs inside retrain(); a losing candidate is discarded")
print("and the incumbent keeps serving.")

## 5. Transfer learning: the sparse-district version of the same problem

A district with too little history cannot be refitted at all. Rather than
transplanting a model from an ecologically different district — the mistake
shortcoming #8 describes — the platform borrows from *similar* districts and
applies a two-parameter local calibration.

In [ ]:
from src.models.transfer_learning import (
    build_transfer_plan, find_donors, fit_local_calibration, similarity_matrix,
)

similarity = similarity_matrix(FULL_REGION)
for district in ["Moshi MC", "Sengerema", "Kinondoni"]:
    similar = similarity[district].drop(district).nlargest(4)
    print(f"{district:14} most similar: " +
          ", ".join(f"{d} ({v:.2f})" for d, v in similar.items()))

print("\nSimilarity uses latitude, longitude, population, density, WASH coverage and")
print("urbanicity - so a highland council borrows from other highland councils, not")
print("from whichever lowland district happens to be geographically nearest.")

plan = build_transfer_plan(matrix, REGION, min_rows=10_000)   # force the borrowing path
display(plan.summary())

In [ ]:
rng = np.random.default_rng(11)
pooled_predictions = rng.normal(60, 15, 30)
local_actuals = 1.4 * pooled_predictions + 8 + rng.normal(0, 3, 30)   # locally biased

calibration = fit_local_calibration(pooled_predictions, local_actuals, "SparseDistrict")
print(calibration.describe())
print(f"\nR2 before calibration : {calibration.r2_before:+.3f}")
print(f"R2 after calibration  : {calibration.r2_after:+.3f}")
print("\nTwo parameters instead of a hundred - which is what makes local adaptation")
print("feasible on a council with a year of patchy reporting.")

## 6. Running this in production

The retraining cycle is a scheduled job, not a notebook step:

```bash
# check every disease, refit and promote where warranted
python -c "from src.models.auto_retrain import run_retraining_cycle; \
           print(run_retraining_cycle())"

# or via the API
curl -X POST localhost:8000/admin/retrain -H 'Content-Type: application/json' -d '{}'

# inspect the current drift verdict without refitting
curl localhost:8000/admin/drift/malaria
```

**What to monitor:** if `should_retrain` fires on drift more than once or twice a
year for the same disease, the problem is usually upstream — a changed reporting
practice or a broken feed — and refitting is treating the symptom. Check
`GET /data/status` first.

In [ ]:
state = retrainer.load_state()
print("persisted retraining state:")
for key, value in state.items():
    if key == "residuals":
        print(f"  residuals        : {len(value)} values retained (rolling)")
    else:
        print(f"  {key:17}: {value}")

## Takeaways

* Both detectors catch a step change within tens of observations and neither
  fires on a stationary stream.
* Retraining triggers on **cadence or drift**, whichever comes first.
* Promotion is gated on held-out performance, so the loop cannot degrade itself.
* Sparse districts borrow structure from ecologically similar donors and calibrate
  locally, rather than inheriting a transplanted model wholesale.

Next: **`07_alerting_and_interventions.ipynb`** — from a forecast to an action,
and back again.